In [1]:
from datasets import load_dataset
from tqdm.notebook import tqdm
import matplotlib.pyplot as plt
from transformers import AutoTokenizer
import torch
from torch import nn
import numpy as np
from torch import nn
import torch
from torch.utils.data import DataLoader

# Transformer from scratch

In the two following sessions, we are going to code a Transformer model from scratch. For this first session, we focus on Attention for text classification. The goal is to develop a modular code, increasing the difficulty step by step.

# Attention

In this part, we will build an attention module, step by step. To have modular code, it is better to use `class` to code our module. We will mainly follow the course to get a good intuition of how attention works.

:red_circle: **WARNING** :red_circle: **You are not allowed to use any existing pre-built torch module, like `nn.Linear`.** You can only use pytorch `Tensor` and the module you create.


In [ ]:
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

In [ ]:
class_names = ["World", "Sports", "Business", "Sci/Tech"]

In [ ]:
dataset = load_dataset("sh0416/ag_news")
dataset = dataset["train"].select(range(1000))
dataset = dataset.shuffle(seed=42)
dataset = dataset.train_test_split(test_size=0.3, seed=1234)
dataset["validation"] = dataset["test"]

print(dataset)


def preprocess_text(x):

    ids = tokenizer(x["description"], truncation=True, max_length=256, padding=False)[
        "input_ids"
    ]
    return {"input_ids": ids, "label": x["label"] - 1}


# Clean the dataset and tokenize it directly
dataset = dataset.map(preprocess_text)

In [ ]:
class DataCollator:
    def __init__(self, tokenizer, max_len=128):
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __call__(self, batch):
        # Tokenize the texts
        labels = [example["label"] for example in batch]
        tokenized_texts = [example["input_ids"] for example in batch]
        # Pad the tokenized texts
        max_len = max(len(text) for text in tokenized_texts)
        padded_texts = [
            text + [self.tokenizer.pad_token_id] * (max_len - len(text))
            for text in tokenized_texts
        ]
        pad_mask = [
            [1] * len(text) + [0] * (max_len - len(text)) for text in tokenized_texts
        ]
        return {
            "input_ids": torch.tensor(padded_texts),
            "pad_mask": torch.tensor(pad_mask),
            "labels": torch.tensor(labels),
        }

In [ ]:
def validation_step(valid_dataloader, model, criterion):
    n_valid = len(valid_dataloader.dataset)
    model.eval()
    total_loss = 0.0
    correct = 0
    n_iter = 0
    with torch.no_grad():
        for batch in valid_dataloader:
            input_ids = batch["input_ids"].to(DEVICE)
            labels = batch["labels"].to(DEVICE)
            pad_mask = batch["pad_mask"].to(DEVICE)
            output = model(input_ids, pad_mask)
            loss = criterion(output, labels)
            total_loss += loss.item()
            correct += (output.argmax(axis=-1) == labels).sum().item()
            n_iter += 1
    return total_loss / n_iter, correct / n_valid


def train_one_epoch(train_dataloader, model, optimizer, criterion):
    model.train()
    total_loss = 0.0
    correct = 0
    n_train = len(train_dataloader.dataset)
    n_iter = 0
    for batch in train_dataloader:
        optimizer.zero_grad()
        input_ids = batch["input_ids"].to(DEVICE)
        labels = batch["labels"].to(DEVICE)
        pad_mask = batch["pad_mask"].to(DEVICE)
        class_scores = model(input_ids, pad_mask)  # (B, 4)

        loss = criterion(class_scores, labels)  # scalaire (1,)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        correct += (class_scores.argmax(axis=-1) == labels).sum().item()
        n_iter += 1

    return total_loss / n_iter, correct / n_train


def train(model, train_dataloader, valid_dataloader, lr=0.01, n_epochs=5):
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    # Track training loss, training accuracy, validation loss and validation accuracy and plot in the end
    train_losses = []
    train_accuracies = []
    valid_losses = []
    valid_accuracies = []
    model.to(DEVICE)
    for epoch in tqdm(range(n_epochs)):
        train_loss, train_accuracy = train_one_epoch(
            train_dataloader, model, optimizer, criterion
        )
        valid_loss, valid_accuracy = validation_step(valid_dataloader, model, criterion)
        print(
            f"Epoch {epoch + 1}: train_loss: {train_loss:.4f}, train_accuracy: {train_accuracy:.4f}, valid_loss: {valid_loss:.4f}, valid_accuracy: {valid_accuracy:.4f}"
        )
        train_losses.append(train_loss)
        train_accuracies.append(train_accuracy)
        valid_losses.append(valid_loss)
        valid_accuracies.append(valid_accuracy)

    plt.figure(figsize=(12, 6))
    plt.subplot(1, 2, 1)
    plt.plot(train_losses, label="train loss")
    plt.plot(valid_losses, label="valid loss")
    plt.legend()
    plt.subplot(1, 2, 2)
    plt.plot(train_accuracies, label="train accuracy")
    plt.plot(valid_accuracies, label="valid accuracy")
    plt.legend()

## Attention-based classification model

The first step since we have access to text in a tokenized form is to use "static" word embeddings.

### Embeddings


🚧 **TODO** 🚧

Write a class named "Embeddings" that will take as input a list of ids and return associated vectors. The ids are integer and their range is given by the vocabulary size. This last quantity directly derives from the choice we made when building the tokenizer.

In [ ]:
class Embeddings(nn.Module):
    def __init__(self, vocab_size, d):
        # TODO

    def forward(self, x):
        # TODO


### Attention

Here we will write a 'AttentionBasedClassifier' `class` that will take as input a list of vectors and return a list of probabilities over the possible classes. The classes should be contextualized with the input vectors, using attention.

🚧 **TODO** 🚧

Write a class named "ClassAttention" that will take as input a list of word vectors, contextualize the embeddings over the classes and return a list of probabilities over classes.

**HINT** Split the architectures into modular blocks.

In [ ]:
class Linear(nn.Module):
    def __init__(self, d_in, d_out, add_bias=True):
        # TODO

    def forward(self, x):
        # TODO

class Attention(nn.Module):
    def __init__(self, d):
        # TODO

    def forward(self, x1, x2):
        # TODO



class Embeddings(nn.Module):
    def __init__(self, vocab_size, d):
        # TODO
        super().__init__()
        self.embeddings = nn.Parameter(torch.randn(vocab_size, d))

    def forward(self, x):
        # x is a batch of ids (B, L)
        return self.embeddings[x]


### ----- TODO of this TP --------###
class ClassAttention(nn.Module):
    def __init__(self, d, n_classes, vocab_size):
        super().__init__()
        # TODO
        self.d = d
        self.n_classes = n_classes


        # 1/ Learnable class embeddings (queries)
        self.class_embeddings = nn.Parameter(
            torch.randn(n_classes, d)
        )

        # 2/ Word embedding layer (keys)
        # self.embedding = nn.Embedding(vocab_size, d)
        self.embedding = Embeddings(vocab_size, d)


        # 3/ Projection layers (mappings Q, K, V)
        self.q_proj = nn.Linear(d, d, bias=False) # (D, D)
        self.k_proj = nn.Linear(d, d, bias=False)
        self.v_proj = nn.Linear(d, d, bias=False)

        # 4/ Final scoring layer (per class)
        self.output_layer = nn.Linear(d, 1, bias=False)

    def forward(self, x):
        # TODO

        # Embed tokens
        x = self.embedding(x)   # (B, L, D) B: batch size, L: longeur du texte, D: dimension of embedding
        B, L, D = x.shape

        # Keys & values from words
        k = self.k_proj(x)      # k_j = K.x_j  (B, L, D)
        v = self.v_proj(x)      # v_j = V.x_j  (B, L, D)

        # Expand class embeddings across batch
        # C: nombre de classes
        #   taille (C, D) -> (B, C, D)
        class_emb = self.class_embeddings.unsqueeze(0).expand(B, C, D)
        # NOT q_i = Q.x_i, since we do not work on self-attention # (B, C, D)
        # but q_i = Q. (class_emb)_i  # apply Q on the last dim of (class_emb)
        q = self.q_proj(class_emb) # (B, C, D)

        # S <- QK'/sqrt{d}
        # Attention: classes attend to words
        #  the multiplications below are: s_ij = <q_i, k_j> for every sample b in batch
        scores = q @ k.transpose(1, 2)   # (B,C,D) x (B,D,L) -> (B,C,L)
        scores = scores / math.sqrt(D)

        attention = F.softmax(scores, dim=2)  # over words
        # Context per class (Y)
        context = attention @ v  #  (B,C,L) x (B,L,D) -> (B, C, D)

        # Scalar score per class
        #  muliplication " W_O Y"   (B,C,D) x (D, 1) -> (B, C), classe_output for each sample b
        logits = self.output_layer(context).squeeze(-1)  #  (B, C)
        # Probabilities over classes
        probs = F.softmax(logits, dim=1)

        return probs

In [ ]:
# Try the model on a simple input:
model = ClassAttention(d=50, n_classes=4, vocab_size=10000)
x = torch.randint(0, 10000, (2, 10))
output = model(x)
print(output.shape)

In [ ]:
# batch_size = 4
batch_size = 32
n_train = len(dataset["train"])
n_valid = len(dataset["test"])
data_collator = DataCollator(tokenizer)
train_dataloader = DataLoader(
    dataset["train"], batch_size=batch_size, collate_fn=data_collator, shuffle=True
)
valid_dataloader = DataLoader(
    dataset["test"], batch_size=batch_size, collate_fn=data_collator, shuffle=True
)

model = ClassAttention(d=10, n_classes=4, vocab_size=len(tokenizer))
train(
    model,
    train_dataloader=train_dataloader,
    valid_dataloader=valid_dataloader,
)

🚧 **TODO** 🚧

Update the code above such that it takes as argument an `attention_mask` for padding.

Use this `attention_mask` to modify the attention computation such that padding tokens do not interfere in the computation.

In [ ]:
# TODO
INF = 1e10


class MaskedAttention(nn.Module):
    def __init__(self, d):
        # TODO

    def forward(self, x1, x2, pad_mask):
        # TODO


class MaskedClassAttention(ClassAttention):
    def __init__(self, d, n_classes, vocab_size):
        # TODO

    def forward(self, x, attention_mask):
        # TODO

🔴 **TEST**

In [ ]:
model = MaskedClassAttention(d=50, n_classes=4, vocab_size=10000)

seq_len = 10
x_without_pad = torch.randint(0, 10000, (1, seq_len))
attention_mask = torch.ones(1, seq_len)
out_without_pad = model(x_without_pad, attention_mask)

x_with_pad = torch.randint(0, 10000, (1, 2 * seq_len))
attention_mask = torch.ones(1, 2 * seq_len)
attention_mask[:, seq_len:] = 0
out_with_pad = model(x_with_pad, attention_mask)

assert out_without_pad.shape == out_with_pad.shape

In [ ]:
batch_size = 4
n_train = len(dataset["train"])
n_valid = len(dataset["test"])
data_collator = DataCollator(tokenizer)
train_dataloader = DataLoader(
    dataset["train"], batch_size=batch_size, collate_fn=data_collator, shuffle=True
)
valid_dataloader = DataLoader(
    dataset["test"], batch_size=batch_size, collate_fn=data_collator, shuffle=True
)

model = MaskedClassAttention(d=10, n_classes=4, vocab_size=len(tokenizer))
train(
    model,
    train_dataloader=train_dataloader,
    valid_dataloader=valid_dataloader,
)

# Self-attention

In this part, we will update the function to add a module of self-attention beforet the class attention.

In [ ]:
class AlmostTransformer(ClassAttention):
    def __init__(self, d, n_classes, vocab_size):
        # TODO

    def forward(self, x, attention_mask):
        # TODO